<div style="background:#F3E8FF; border-left:7px solid #7E22CE; padding:24px; color:#3B0764;">
<h1>PBrownWeek7 | Precious’s Movie Night</h1>
<p><strong>Precious Brown · IS 362 · Data Acquisition and Management</strong></p>
<p>Week 7: Data Aggregation · Movie ratings, normalization, and standardization</p>
</div>

This project brings a movie-night theme to my Information Systems coursework. I compare how five reviewers rate six films, then explore how changing the rating scale affects the results. The purple presentation reflects my personal style, while the analysis focuses on clear, reproducible pandas operations.

**Data disclosure:** All five reviewers and all ratings are fictional, using the assignment’s allowance for imaginary friends. These are not survey responses or claims about my own viewing history. Each blank CSV cell means the fictional reviewer has not seen that movie.

**Run instructions:** Keep `PBrownWeek7.ipynb` and `PBrownWeek7Ratings.csv` in the same folder. Open the notebook in Jupyter and select **Restart Kernel and Run All Cells**. In Colab, upload the CSV to the session before running. Markdown explanations belong in Markdown cells; Python belongs in code cells.

<h2 style="color:#6B21A8; border-bottom:2px solid #D8B4FE; padding-bottom:8px;">PBrownWeek7 · 1. Source Data and Loading</h2>

The source CSV stores one reviewer per row and one movie per column. Ratings use whole numbers from **1 (lowest) to 5 (highest)**. The six selections combine musical storytelling, animation, and adventure for a varied movie-night lineup.

The films are 2025 releases listed on [Box Office Mojo’s 2025 worldwide chart](https://www.boxofficemojo.com/year/world/2025/), accessed October 18, 2026. This source supports film selection only; it does not supply the fictional ratings.

I load the saved observations with `pd.read_csv()` and use `User` as the index. Empty CSV fields become `NaN`, pandas’ missing-value marker. I do not replace missing ratings with zero because zero is outside the rating scale and would incorrectly penalize unseen movies.

In [1]:
import pandas as pd
from IPython.display import display

# The relative path keeps the project portable when both files stay together.
ratings = pd.read_csv("PBrownWeek7Ratings.csv", index_col="User")

# A small presentation helper keeps the analysis itself straightforward.
def purple_table(frame, caption):
    return (frame.style
            .format(precision=2, na_rep="Not rated")
            .set_caption(caption)
            .set_table_styles([
                {"selector": "caption", "props": [("color", "#6B21A8"),
                                                     ("font-weight", "bold"),
                                                     ("text-align", "left")]},
                {"selector": "th", "props": [("background-color", "#E9D5FF"),
                                                ("color", "#3B0764")]},
                {"selector": "td", "props": [("background-color", "#FAF5FF"),
                                                ("color", "#3B0764")]}
            ]))

display(purple_table(ratings, "Precious’s Movie Night | Original ratings"))

,Wicked: For Good (2025),Zootopia 2 (2025),Lilo & Stitch (2025),A Minecraft Movie (2025),How to Train Your Dragon (2025),Superman (2025)
User,,,,,,
Avery,5.00,4.00,5,2.00,4.00,Not rated
Jordan,3.00,3.00,2,1.00,Not rated,4.00
Morgan,5.00,5.00,4,Not rated,5.00,3.00
Riley,4.00,Not rated,3,2.00,4.00,5.00
Taylor,Not rated,4.00,5,3.00,4.00,2.00


Before calculating averages, I check the assignment’s required dimensions and confirm that observed ratings are integers between 1 and 5. A missing-value count shows which films were not seen by everyone.

In [2]:
observed_ratings = ratings.stack()
assert ratings.shape[0] >= 5, "At least five reviewers are required."
assert ratings.shape[1] == 6, "Exactly six movies are required."
assert ratings.index.is_unique, "Reviewer names must be unique."
assert observed_ratings.between(1, 5).all(), "Ratings must be from 1 to 5."
assert (observed_ratings % 1 == 0).all(), "Use whole-number ratings."
assert ratings.isna().any().any(), "At least one movie must have an unseen entry."
assert ratings.notna().any(axis=1).all(), "Each reviewer needs a rating."
assert ratings.notna().any(axis=0).all(), "Each movie needs a rating."

print(f"{len(ratings)} reviewers, {len(ratings.columns)} movies")
print(f"{ratings.count().sum()} observed ratings; {ratings.isna().sum().sum()} unseen entries")
display(purple_table(ratings.isna().sum().to_frame("Unseen entries"),
                     "Missing ratings by movie"))

5 reviewers, 6 movies
25 observed ratings; 5 unseen entries


,Unseen entries
Wicked: For Good (2025),1
Zootopia 2 (2025),1
Lilo & Stitch (2025),0
A Minecraft Movie (2025),1
How to Train Your Dragon (2025),1
Superman (2025),1


<h2 style="color:#6B21A8; border-bottom:2px solid #D8B4FE; padding-bottom:8px;">PBrownWeek7 · 2. Average Original Ratings</h2>

`axis=1` calculates across each reviewer’s row; `axis=0` calculates down each movie’s column. Pandas excludes missing values from means by default. I also report rating counts so that averages based on different amounts of information are not mistaken for equally supported results. Rounding affects display only.

In [3]:
user_averages = ratings.mean(axis=1)
movie_averages = ratings.mean(axis=0)

user_summary = pd.DataFrame({
    "Movies rated": ratings.count(axis=1),
    "Average rating": user_averages
})
movie_summary = pd.DataFrame({
    "Reviewer count": ratings.count(axis=0),
    "Average rating": movie_averages
})

display(purple_table(user_summary, "Original averages | Each reviewer"))
display(purple_table(movie_summary, "Original averages | Each movie"))

,Movies rated,Average rating
User,,
Avery,5,4.00
Jordan,5,2.60
Morgan,5,4.40
Riley,5,3.60
Taylor,5,3.60


,Reviewer count,Average rating
Wicked: For Good (2025),4,4.25
Zootopia 2 (2025),4,4.00
Lilo & Stitch (2025),5,3.80
A Minecraft Movie (2025),4,2.00
How to Train Your Dragon (2025),4,4.25
Superman (2025),4,3.50


<h2 style="color:#6B21A8; border-bottom:2px solid #D8B4FE; padding-bottom:8px;">PBrownWeek7 · 3. Normalize Each Reviewer’s Ratings</h2>

I use **per-reviewer min-max normalization**, based on each reviewer’s observed minimum and maximum:

$$x_{normalized} = \frac{x - \min(x_{user})}{\max(x_{user}) - \min(x_{user})}$$

For a reviewer with varied ratings, the lowest observed score becomes 0 and the highest becomes 1. This is different from applying the fixed 1–5 scale formula `(x - 1) / 4` to everyone. The transformation is performed **within rows**, not within movie columns.

If a reviewer gives only identical ratings, the range is zero and min-max scaling is undefined. I replace that denominator with `NaN`, leaving that reviewer’s transformed values undefined instead of inventing preference differences. This also handles a reviewer with only one observed rating. The current example has no such rows. Original unseen entries remain missing.

In [4]:
user_minimum = ratings.min(axis=1)
user_maximum = ratings.max(axis=1)
user_range = user_maximum - user_minimum
safe_range = user_range.replace(0, float("nan"))

# axis=0 aligns the Series of per-user values with DataFrame row labels.
normalized = ratings.sub(user_minimum, axis=0).div(safe_range, axis=0)

display(purple_table(normalized, "New DataFrame | Normalized ratings"))
display(purple_table(normalized.mean(axis=1).to_frame("Normalized average"),
                     "Normalized averages | Each reviewer"))
display(purple_table(normalized.mean(axis=0).to_frame("Normalized average"),
                     "Normalized averages | Each movie"))

,Wicked: For Good (2025),Zootopia 2 (2025),Lilo & Stitch (2025),A Minecraft Movie (2025),How to Train Your Dragon (2025),Superman (2025)
User,,,,,,
Avery,1.00,0.67,1.00,0.00,0.67,Not rated
Jordan,0.67,0.67,0.33,0.00,Not rated,1.00
Morgan,1.00,1.00,0.50,Not rated,1.00,0.00
Riley,0.67,Not rated,0.33,0.00,0.67,1.00
Taylor,Not rated,0.67,1.00,0.33,0.67,0.00


,Normalized average
User,
Avery,0.67
Jordan,0.53
Morgan,0.70
Riley,0.53
Taylor,0.53


,Normalized average
Wicked: For Good (2025),0.83
Zootopia 2 (2025),0.75
Lilo & Stitch (2025),0.63
A Minecraft Movie (2025),0.08
How to Train Your Dragon (2025),0.75
Superman (2025),0.50


<h2 style="color:#6B21A8; border-bottom:2px solid #D8B4FE; padding-bottom:8px;">PBrownWeek7 · 4. Extra Credit: Standardize Each Reviewer’s Ratings</h2>

For the extra credit, I create another DataFrame using **per-reviewer z-scores**:

$$z = \frac{x - \mu_{user}}{\sigma_{user}}$$

I use `std(axis=1, ddof=0)`, the population standard deviation of each reviewer’s observed scores, to describe this collected set rather than estimate a larger population’s variance. Pandas defaults to `ddof=1`, so the explicit choice matters.

Positive z-scores are above that reviewer’s mean; negative scores are below it. Zero means the rating equals the reviewer’s mean. Standardized ratings are not restricted to 0–1. Each valid reviewer’s mean should be approximately zero, with tiny floating-point differences possible. A zero standard deviation is handled as undefined, just like a zero range above.

In [5]:
user_std = ratings.std(axis=1, ddof=0)
safe_std = user_std.replace(0, float("nan"))
standardized = ratings.sub(user_averages, axis=0).div(safe_std, axis=0)

display(purple_table(standardized, "New DataFrame | Standardized ratings"))
display(purple_table(standardized.mean(axis=1).to_frame("Standardized average"),
                     "Standardized averages | Each reviewer"))
display(purple_table(standardized.mean(axis=0).to_frame("Standardized average"),
                     "Standardized averages | Each movie"))

,Wicked: For Good (2025),Zootopia 2 (2025),Lilo & Stitch (2025),A Minecraft Movie (2025),How to Train Your Dragon (2025),Superman (2025)
User,,,,,,
Avery,0.91,0.00,0.91,-1.83,0.00,Not rated
Jordan,0.39,0.39,-0.59,-1.57,Not rated,1.37
Morgan,0.75,0.75,-0.50,Not rated,0.75,-1.75
Riley,0.39,Not rated,-0.59,-1.57,0.39,1.37
Taylor,Not rated,0.39,1.37,-0.59,0.39,-1.57


,Standardized average
User,
Avery,0.00
Jordan,-0.00
Morgan,-0.00
Riley,-0.00
Taylor,-0.00


,Standardized average
Wicked: For Good (2025),0.61
Zootopia 2 (2025),0.38
Lilo & Stitch (2025),0.12
A Minecraft Movie (2025),-1.39
How to Train Your Dragon (2025),0.38
Superman (2025),-0.14


<h2 style="color:#6B21A8; border-bottom:2px solid #D8B4FE; padding-bottom:8px;">PBrownWeek7 · 5. Compare Results</h2>

The side-by-side table shows all three movie averages with their observed rating counts. The columns use different units, so their numerical sizes should not be compared directly. Their ordering helps show how adjusting for individual rating habits can change the group’s apparent preferences.

In [6]:
comparison = pd.DataFrame({
    "Ratings received": ratings.count(),
    "Original average (1–5)": movie_averages,
    "Normalized average (0–1)": normalized.mean(),
    "Standardized average (z-score)": standardized.mean()
})
display(purple_table(comparison, "Movie averages | Three perspectives"))

print("Highest original mean (first movie if tied):", movie_averages.idxmax())
print("Highest normalized mean:", normalized.mean().idxmax())
print("Highest standardized mean:", standardized.mean().idxmax())

Highest original mean (first movie if tied): Wicked: For Good (2025)
Highest normalized mean: Wicked: For Good (2025)
Highest standardized mean: Wicked: For Good (2025)


,Ratings received,Original average (1–5),Normalized average (0–1),Standardized average (z-score)
Wicked: For Good (2025),4,4.25,0.83,0.61
Zootopia 2 (2025),4,4.00,0.75,0.38
Lilo & Stitch (2025),5,3.80,0.63,0.12
A Minecraft Movie (2025),4,2.00,0.08,-1.39
How to Train Your Dragon (2025),4,4.25,0.75,0.38
Superman (2025),4,3.50,0.50,-0.14


<h2 style="color:#6B21A8; border-bottom:2px solid #D8B4FE; padding-bottom:8px;">PBrownWeek7 · 6. Conclusion</h2>

**What the example shows.** The dataset contains 25 observed ratings and five unseen entries across five fictional reviewers and six movies. Jordan’s original average is 2.60, while Morgan’s is 4.40, illustrating different levels of scoring generosity. Wicked: For Good and How to Train Your Dragon tie for the highest original average at 4.25. Wicked: For Good has the highest average after min-max normalization, approximately 0.83, and after standardization, approximately 0.61. The transformations break the original tie. The change shows that a movie’s absolute average and its position relative to each reviewer’s usual ratings can tell different stories.

**Advantages of normalization.** Putting each reviewer’s observed scores on a 0–1 scale makes relative preferences easier to compare when reviewers use different portions of the rating scale. For my movie-night example, someone who usually gives lower ratings can still contribute a strong relative preference. The result may be useful as an input to a recommendation system, although this notebook does not build or evaluate one.

**Disadvantages of normalization.** Min-max scaling removes the original meaning of a star rating. A normalized 1 means that reviewer’s highest observed score, which might be a 4 rather than a 5. It can also exaggerate small differences when someone uses a narrow range. The result depends on the observed minimum and maximum, so adding another rating can change earlier normalized values. Identical ratings cannot be min-max normalized under this formula.

**What standardization adds.** Z-scores account for both a reviewer’s average and the spread of their ratings. They show how far a movie sits above or below that person’s usual score in standard-deviation units. However, negative values are less intuitive than stars, and small or low-variation samples can produce unstable comparisons. A mean of zero for each standardized reviewer is a mathematical result, not evidence that everyone liked the movies equally.

**My takeaway.** I would retain the original ratings for a clear description of satisfaction and use transformed ratings when comparing relative preferences. Missing ratings stay missing; they are not dislikes. Each movie was rated by a different subset of reviewers, and neither transformation corrects that selection issue. Because this is a small fictional dataset, the results demonstrate pandas techniques rather than establish real audience preferences. In my Information Systems work, explaining these limitations is as important as calculating the numbers.

<h2 style="color:#6B21A8; border-bottom:2px solid #D8B4FE; padding-bottom:8px;">PBrownWeek7 · Submission Checklist</h2>

- Upload `PBrownWeek7.ipynb` and `PBrownWeek7Ratings.csv` to the same GitHub folder.
- Include `README.md` and `requirements.txt` for reproducibility.
- Open the notebook on GitHub and confirm that saved outputs are visible.
- Submit the GitHub link as directed by the instructor.

**Requirement coverage:** CSV loading appears in Section 1; original user/movie averages in Section 2; a new normalized DataFrame and its averages in Section 3; a new standardized DataFrame and its averages in Section 4; the required text conclusion in Section 6.

**Presentation note:** HTML headings and pandas tables use purple styling in Jupyter. GitHub may sanitize some inline styles, but the section text, code, and saved results remain readable.